# Aplicación simple con LangChain

_Qué es LangChain, sus abstracciones, LCEL y un mini-asistente de Q&A sobre nuestra propia documentación_

**Módulo 3 — Introducción a AI Engineering** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![AI Engineering](assets/header.png)

## 1. ¿Qué es LangChain?

[LangChain](https://www.langchain.com/) es un **framework para construir aplicaciones con LLMs**. No es un modelo: es la "tubería" que conecta el modelo con prompts, fuentes de datos, herramientas, memoria y otros componentes.

Su propuesta de valor:
- **Abstracciones comunes** sobre múltiples proveedores (OpenAI, Anthropic, Gemini, Ollama, HuggingFace, …) → cambias el modelo cambiando una línea.
- **Composición declarativa** — encadenas componentes con el operador `|` (LCEL — LangChain Expression Language).
- **Ecosistema** — 600+ integraciones con vector stores, document loaders, herramientas, observabilidad.
- **Patrones listos** — RAG, agents, memory, structured output ya están resueltos.

## 2. Cuándo SÍ y cuándo NO usar LangChain

| Conviene | No conviene |
|---|---|
| Tu app encadena varios pasos (retriever → prompt → LLM → parser) | Una sola llamada `chat.completions.create` resuelve el problema |
| Quieres poder cambiar de proveedor sin reescribir | Estás 100% comprometido con un solo SDK |
| Necesitas integrar muchas fuentes (PDF, SQL, vector store, APIs) | Tu fuente es trivial (un string en memoria) |
| Estás haciendo RAG, agents, memory, tool use serios | Estás haciendo un wrapper fino sobre un endpoint |
| Quieres tracing/eval listos (LangSmith) | No te pesa montar tu propio logging |

> 💡 Regla práctica: **arranca sin LangChain** (con el SDK directo del proveedor). Adóptalo cuando sientas que estás reinventando sus abstracciones.

## 3. Las abstracciones core

| Abstracción | Qué representa | Ejemplo |
|---|---|---|
| `ChatModel` | un LLM con interfaz de chat | `ChatOpenAI(model='gpt-4o-mini')` |
| `Embeddings` | un modelo de embeddings | `OpenAIEmbeddings(model='text-embedding-3-small')` |
| `PromptTemplate` / `ChatPromptTemplate` | plantilla con variables | `'Resume: {texto}'` |
| `OutputParser` | parsea la salida del LLM | `JsonOutputParser`, `PydanticOutputParser` |
| `Runnable` | cualquier paso encadenable con `|` | `prompt | llm | parser` |
| `Retriever` | recupera documentos relevantes | `vectorstore.as_retriever()` |
| `VectorStore` | índice de embeddings | `Chroma`, `FAISS`, `Pinecone` |
| `Document` | texto + metadata | `Document(page_content=..., metadata=...)` |
| `Tool` / `Agent` | función ejecutable + lazo de razonamiento | `@tool def get_weather(...)` |
| `Memory` (legacy) / `History` | mantener contexto entre turnos | `RunnableWithMessageHistory` |

### LCEL — LangChain Expression Language

La sintaxis declarativa de LangChain. Encadenas componentes con `|`:

```python
chain = prompt | llm | output_parser
chain.invoke({'tema': 'embeddings'})
```

Ventajas: streaming, batching, async y observabilidad **gratis** sin tocar tu código.

## 4. Setup

```bash
uv add langchain langchain-openai langchain-community langchain-text-splitters
# Para vector store local:
uv add langchain-chroma
```

Necesitas tu `OPENAI_API_KEY` en el entorno (igual que en notebooks anteriores).

In [ ]:
# Cargar API key desde .env si existe
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env')
except ImportError:
    pass

if not os.environ.get('OPENAI_API_KEY'):
    print('⚠️  No hay OPENAI_API_KEY. Las celdas de ejemplo no se ejecutarán.')
else:
    print('✅ API key encontrada.')

## 5. Cadena más simple posible — Prompt + LLM + Parser

Vamos a construir el "Hello World" de LangChain: una cadena que toma un tema y devuelve una explicación corta. Luego añadiremos piezas.

In [ ]:
# uv add langchain langchain-openai
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'Eres un profesor que explica en máximo 2 frases para principiantes.'),
    ('user',   'Explica qué es {tema}.'),
])

parser = StrOutputParser()

# La magia de LCEL: encadenar con `|`
chain = prompt | llm | parser

print(chain.invoke({'tema': 'el algoritmo K-Means'}))
print('---')
print(chain.invoke({'tema': 'la arquitectura Transformer'}))

**Lo que acabamos de obtener gratis:**
- `chain.batch([...])` — procesar varias entradas en paralelo.
- `chain.stream(...)` — recibir tokens a medida que se generan.
- `await chain.ainvoke(...)` — versión asíncrona.
- Observabilidad si configuras LangSmith (`LANGCHAIN_TRACING_V2=true`).

Sin LangChain tendrías que implementar batching, streaming y tracing tú mismo.

## 6. Salida estructurada con Pydantic

Patrón muy útil: forzar al LLM a devolver un objeto con campos concretos. Usamos un `PydanticOutputParser` o el método nativo `with_structured_output`.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class ResumenPaper(BaseModel):
    titulo:        str   = Field(description='Título del paper')
    area:          Literal['NLP', 'Visión', 'RL', 'Tabular', 'Otro']
    contribucion:  str   = Field(description='Contribución principal en una frase')
    puntuacion:    int   = Field(ge=1, le=5, description='Cuán importante es del 1 al 5')

# El método with_structured_output devuelve directamente el objeto Pydantic
llm_estructurado = ChatOpenAI(model='gpt-4o-mini', temperature=0).with_structured_output(ResumenPaper)

abstract = '''
We propose a new architecture, the Transformer, based solely on attention mechanisms,
dispensing with recurrence and convolutions entirely. Experiments on machine translation
show superior quality while being more parallelizable and requiring less time to train.
'''

resultado = llm_estructurado.invoke(f'Resume este abstract: {abstract}')
print(type(resultado).__name__)
print(f'Título      : {resultado.titulo}')
print(f'Área        : {resultado.area}')
print(f'Contribución: {resultado.contribucion}')
print(f'Puntuación  : {resultado.puntuacion}/5')

## 7. Aplicación — mini RAG sobre nuestra propia documentación

Un caso típico: queremos un asistente que responda preguntas usando el contenido de **nuestros propios documentos** (sin alucinar). El patrón es **RAG (Retrieval-Augmented Generation)**:

```
Pregunta → Embedding → Buscar K docs más similares → Pasarlos como contexto al LLM → Respuesta
```

Vamos a indexar el README del propio repo y preguntarle cosas.

In [ ]:
# uv add langchain-chroma langchain-community langchain-text-splitters
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 1. Cargar el README del repo
docs = TextLoader('../README.md').load()

# 2. Trocearlo en chunks pequeños (los LLMs trabajan mejor con contexto acotado)
splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
chunks = splitter.split_documents(docs)
print(f'Documentos: {len(docs)} | Chunks: {len(chunks)}')

# 3. Embeddings + vector store local (Chroma en memoria)
embeddings   = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore  = Chroma.from_documents(chunks, embeddings)
retriever    = vectorstore.as_retriever(search_kwargs={'k': 4})

# 4. Probemos solo el retriever
hits = retriever.invoke('¿Qué dataset usa el módulo de clustering?')
for i, h in enumerate(hits, 1):
    print(f'--- Hit {i} ---')
    print(h.page_content[:200], '...\n')

In [ ]:
# 5. Cadena completa: pregunta → retriever → prompt con contexto → LLM → respuesta
from langchain_core.runnables import RunnablePassthrough

prompt_rag = ChatPromptTemplate.from_messages([
    ('system',
     'Eres un asistente del curso DSRP Machine Learning Engineering. '
     'Responde la pregunta del usuario usando ÚNICAMENTE el contexto que se te da. '
     'Si la respuesta no está en el contexto, di "No lo sé". '
     'Sé conciso (máximo 4 frases).'),
    ('user', 'Contexto:\n{contexto}\n\nPregunta: {pregunta}'),
])

def formatear_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

rag_chain = (
    {'contexto': retriever | formatear_docs, 'pregunta': RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

preguntas = [
    '¿Qué módulos tiene el curso?',
    '¿Qué algoritmo usa el notebook de clustering?',
    '¿Cuál es la receta de la pizza margarita?',  # no está en el README
]
for p in preguntas:
    print(f'❓ {p}')
    print(f'   → {rag_chain.invoke(p)}\n')

**Qué pasó técnicamente:**
1. El retriever buscó los 4 chunks del README más cercanos en embedding a la pregunta.
2. Esos chunks se inyectaron como `{contexto}` en el prompt.
3. El LLM respondió **basándose solo en lo recuperado**.
4. Para preguntas fuera del corpus (la pizza), el sistema dijo "No lo sé" porque así se lo instruimos.

Eso es un sistema RAG en menos de 30 líneas. La misma estructura se escala a miles de PDFs cambiando solo el `DocumentLoader` y el vector store.

## 8. Cambiar de proveedor con una línea

Una de las grandes promesas de LangChain: el resto del código no cambia.

```python
# OpenAI → Anthropic
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model='claude-haiku-4-5')

# OpenAI → Gemini
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

# OpenAI → Ollama local
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2')
```

La cadena `prompt | llm | parser` sigue funcionando exactamente igual.

## 9. Patrones avanzados (mapa rápido)

- **Agents** — el LLM decide qué tools llamar y en qué orden. Hoy se construyen con [LangGraph](https://langchain-ai.github.io/langgraph/) (parte del mismo ecosistema), que da control fino sobre el ciclo de razonamiento.
- **Memory / History** — `RunnableWithMessageHistory` para mantener contexto multi-turno persistido en Redis, SQLite, etc.
- **Streaming + UI** — `chain.stream(...)` se integra trivialmente con FastAPI/Streamlit/Chainlit.
- **Evaluación** — [LangSmith](https://smith.langchain.com/) graba cada llamada, permite hacer evals con LLM-as-judge y comparar versiones de prompts.
- **Async + batch** — `ainvoke`, `abatch` para correr cientos de llamadas en paralelo respetando rate limits.

## 10. Buenas prácticas

1. **Empieza sin LangChain** y muévete cuando sientas la fricción de orquestar varios pasos.
2. **Usa LCEL** (`prompt | llm | parser`), no las clases `LLMChain` / `ConversationChain` antiguas (deprecadas).
3. **Tipa la salida** con `with_structured_output(PydanticModel)` siempre que vayas a parsear.
4. **Versiona tus prompts** — un `PromptTemplate` mal cambiado puede romper producción silenciosamente.
5. **Activa LangSmith desde el día 1** si la app es seria — sin tracing es muy difícil debuggear cadenas largas.
6. **Cuidado con la versión** de LangChain: el ecosistema se mueve rápido, pinea versiones en `pyproject.toml`.

## 11. Referencias

- LangChain docs: https://python.langchain.com/docs/introduction/
- LCEL — LangChain Expression Language: https://python.langchain.com/docs/concepts/lcel/
- LangChain conceptual guide: https://python.langchain.com/docs/concepts/
- LangGraph (agents): https://langchain-ai.github.io/langgraph/
- LangSmith (observabilidad y evals): https://docs.smith.langchain.com/
- Chroma DB: https://docs.trychroma.com/
- Crítica equilibrada: https://www.octomind.dev/blog/why-we-no-longer-use-langchain-for-building-our-ai-agents (lectura recomendada para entender los trade-offs)